Tasa Cambio Venta (BCH)


In [6]:
import requests
import pandas as pd
from Clave import APIKEY
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


#indicadoresIDs=[338,341,344,347,350,353,356,359,362,365,368,371,374]
indicadoresIDs=[620]
FechaInicio="2025-07-08T00:00:00"
#FechaFinal="2022-12-01T00:00:00"
print(FechaInicio)
df_combinado=[]

for indicadorID in indicadoresIDs:
    urlCifras= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}/cifras"
    urlIndicadores= f"https://bchapi-am.azure-api.net/api/v1/indicadores/{indicadorID}"
    
    params1={
        "fechainicio":FechaInicio
        #,"fechafinal":FechaFinal
    }
    
    headers = {
        "clave" : APIKEY,
        "Conten-Type": "application/json",
    }
    
    responseCifras = requests.get(urlCifras, headers=headers, params=params1, verify=False)
    responseIndicadores = requests.get(urlIndicadores, headers=headers,verify=False)
            
    if responseIndicadores.status_code == 200:
        datosIndicadores = responseIndicadores.json()
        #print(json.dumps(datosIndicadores,indent=4))  
        df_Indicadores=pd.DataFrame([datosIndicadores])
        ##print(df_Indicadores.to_string(index=False))
    else:
        print(f"Error {responseIndicadores.status_code}: {responseIndicadores.text}")
    
    if responseCifras.status_code == 200:
        datos1 = responseCifras.json()
        if datos1:
            df_Cifras=pd.DataFrame(datos1)
        else:
            df_Cifras=pd.DataFrame([{
            "Fecha":FechaInicio,
            "Id":0,       
            "IndicadorId":indicadorID,
            "Nombre":df_Indicadores.loc[0,"Nombre"],
            "Descripcion":df_Indicadores.loc[0,"Descripcion"],
            "Valor":0
            }])            
            ##print(df_Cifras.to_string(index=False))
            # Hacer negativo el valor si Indicador_ID es 113 o 155        
        #df_Cifras.loc[
        #    df_Cifras['IndicadorId'].isin([113, 155]), 'Valor'] = -df_Cifras.loc[df_Cifras['IndicadorId'].isin([113, 155]), 'Valor'
        #].abs()
    else:
        print(f"Error {responseCifras.status_code}: {responseCifras.text}")
  

    df_merge=pd.merge(
        df_Cifras,
        df_Indicadores,
        left_on='IndicadorId',
        right_on='Id',
        how='left'
    )

    #print(df_merge)
    #df_merge['Nombre_Indicador']= 'Indice de Precios al Consumidor IPC'  
    df_merge['NOMBRE_INDICADOR']=df_merge['Descripcion_x'].str.split('-').str[-2]
    df_merge['TIPO']=df_merge['Descripcion_x'].apply(lambda x:'-'.join(x.split('-')[3:5]))
    print(indicadorID, end=",")
    df_final=df_merge[['Fecha','Valor']]
    df_combinado.append(df_final)
else:
    print(f"Error {responseCifras.status_code}: {responseCifras.text}") 

df_resultado=pd.concat(df_combinado, ignore_index=True)
df_resultado=df_resultado.rename(columns={
    'Fecha':'DATE_TIME',
    'Valor':'TASA'
})
##df_merge.to_csv('indicador.csv', index=False, encoding='utf-8-sig')
df_resultado['DATE_TIME']=pd.to_datetime(df_resultado['DATE_TIME']).dt.strftime('%Y-%m-%d')
##df_resultado['viernes'] =df_resultado['DATE_TIME'].dt.dayofweek == 4 

2025-07-08T00:00:00
620,Error 200: [{"Id":2730326,"IndicadorId":620,"Nombre":"EC-TCN-01-2","Descripcion":"Tipo de Cambio Nominal - Venta","Fecha":"2025-07-14T00:00:00","Valor":26.3120},{"Id":2724700,"IndicadorId":620,"Nombre":"EC-TCN-01-2","Descripcion":"Tipo de Cambio Nominal - Venta","Fecha":"2025-07-11T00:00:00","Valor":26.3091},{"Id":2719113,"IndicadorId":620,"Nombre":"EC-TCN-01-2","Descripcion":"Tipo de Cambio Nominal - Venta","Fecha":"2025-07-10T00:00:00","Valor":26.3021},{"Id":2713526,"IndicadorId":620,"Nombre":"EC-TCN-01-2","Descripcion":"Tipo de Cambio Nominal - Venta","Fecha":"2025-07-09T00:00:00","Valor":26.2949},{"Id":2707939,"IndicadorId":620,"Nombre":"EC-TCN-01-2","Descripcion":"Tipo de Cambio Nominal - Venta","Fecha":"2025-07-08T00:00:00","Valor":26.2868}]


In [7]:
df_resultado.head(10)

,DATE_TIME,TASA
0,2025-07-14,26.3120
1,2025-07-11,26.3091
2,2025-07-10,26.3021
3,2025-07-09,26.2949
4,2025-07-08,26.2868


In [8]:
#DUPLICADOS
df_resultado=df_resultado.drop_duplicates()
df_resultado['MONEDA']='USD'
df_resultado.head(10)

,DATE_TIME,TASA,MONEDA
0,2025-07-14,26.3120,USD
1,2025-07-11,26.3091,USD
2,2025-07-10,26.3021,USD
3,2025-07-09,26.2949,USD
4,2025-07-08,26.2868,USD


In [9]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'TASA_CAMBIO_VENTA'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}
#columns_types

In [10]:
#-----------------------------############### INSERT ##################------------------------------------#


data_frame=df_resultado
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')

Insertando datos:   0%|          | 0/1 [00:00<?, ?it/s]

Insertando datos: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]

Proceso de insercion completado
